# 🌊 AI Urban Flood & Traffic Congestion Monitor (Official SYNEXAN)
### 100% Giao diện Gradio gốc ở Local (Tab 1, Tab 2, Tab 3 Dual Fusion)
Triển khai trên Kaggle GPU (Tesla T4) + Ngrok Static Domain: `hush-harmless-erased.ngrok-free.dev`.
Phục vụ cho Frontend trên Render.com.

In [ ]:
# 1. Cài đặt các thư viện cần thiết cho Gradio, YOLOv8 và ByteTrack
!pip install -q "huggingface_hub<0.26.0" "gradio==4.44.0" "starlette<0.40.0" "fastapi<0.116.0" "pydantic<2.10.0" pyngrok ultralytics opencv-python-headless pillow pandas lapx nest-asyncio

In [ ]:
# 2. Đồng bộ mã nguồn Backend mới nhất từ GitHub SYNEXAN/FrontendFlood
import os, sys
target_dir = "/kaggle/working/FrontendFlood"
if not os.path.exists(target_dir):
    !git clone https://github.com/SYNEXAN/FrontendFlood.git /kaggle/working/FrontendFlood
else:
    !cd /kaggle/working/FrontendFlood && git pull

os.chdir("/kaggle/working/FrontendFlood/backend")
sys.path.append("/kaggle/working/FrontendFlood/backend")
print("✅ Thư mục backend hiện tại:", os.getcwd())
!ls -la

In [ ]:
# 3. Tự động tải Trọng số mô hình Deep Learning (Flood Seg & Vehicle Detection)
import os, urllib.request, time

os.makedirs("models", exist_ok=True)
models_info = [
    ("models/vietnam_flood_yolov8s_seg.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/vietnam_flood_yolov8s_seg.pt", 20000000),
    ("models/yolov8s_vehicle.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/yolov8s_vehicle.pt", 20000000)
]

for path, url, min_size in models_info:
    if os.path.exists(path) and os.path.getsize(path) >= min_size:
        print(f"✅ Mô hình {path} đã có sẵn ({os.path.getsize(path)} bytes).")
        continue
    for attempt in range(1, 4):
        try:
            print(f"⬇️ Đang tải {path} (lần {attempt}/3)...")
            if os.path.exists(path):
                os.remove(path)
            urllib.request.urlretrieve(url, path)
            if os.path.getsize(path) >= min_size:
                print(f"✅ Tải thành công {path}.")
                break
        except Exception as e:
            print(f"❌ Lỗi tải {path}: {e}")
            time.sleep(2)

print("✅ Sẵn sàng các file mô hình:")
!ls -lh models/

In [ ]:
# 4. Khởi động Ngrok Tunnel & Chạy Gradio App Nguyên Bản (100% Giao diện gốc)
import subprocess
from pyngrok import ngrok

NGROK_AUTHTOKEN = "3LNVtpalQtpB352VIkzga6oB3dg_5qLYQiojiE9bG79gf5Liq"
NGROK_DOMAIN = "hush-harmless-erased.ngrok-free.dev"

if NGROK_AUTHTOKEN:
    try:
        ngrok.set_auth_token(NGROK_AUTHTOKEN)
    except Exception as e:
        print(f"Warning set_auth_token: {e}")

try:
    ngrok.kill()
except Exception:
    pass

public_url = None
try:
    print(f"Connecting via static domain https://{NGROK_DOMAIN} on port 7860...")
    tunnel = ngrok.connect(7860, domain=NGROK_DOMAIN)
    public_url = tunnel.public_url
    print(f"✅ Connected to static domain: {public_url}")
except Exception as e1:
    print(f"Domain connect notice: {e1}")
    tunnel = ngrok.connect(7860)
    public_url = tunnel.public_url

print("=" * 70)
print("🚀 GRADIO WEB APP (100% GIAO DIỆN GỐC KÈM TAB 3 DUAL FUSION) ĐÃ SẴN SÀNG!")
print(f"👉 NGROK PUBLIC URL: {public_url}")
print("=" * 70)

subprocess.run(["python", "app.py"])